In [ ]:
import inspect
import pandas as pd
import sklearn
import re

# ========================
# Configurações iniciais
# ========================
method_path = "sklearn.utils.discovery.all_estimators"   # pode trocar para sklearn.metrics etc.
output_file = "all_estimators.tsv"

# ========================
# Funções auxiliares
# ========================
def parse_doc(doc: str):
    """Divide uma docstring numpy-style em blocos estruturados."""
    sections = {
        "summary": "",
        "parameters": "",
        "attributes": "",
        "see_also": "",
        "examples": "",
        "other": ""
    }
    if not doc:
        return sections

    pattern = re.compile(r"^(Parameters|Attributes|See Also|Examples)\n[-]+\n", re.M)
    parts = pattern.split(doc)

    sections["summary"] = parts[0].strip()

    for i in range(1, len(parts), 2):
        sec = parts[i].lower().replace(" ", "_")
        content = parts[i+1].strip() if i+1 < len(parts) else ""
        if sec in sections:
            sections[sec] = content
        else:
            sections["other"] += f"\n\n{parts[i]}\n{content}"

    return sections

def get_method_func(path: str):
    """Dado um caminho completo, retorna a função/objeto."""
    parts = path.split(".")
    obj = __import__(".".join(parts[:-1]), fromlist=[parts[-1]])
    return getattr(obj, parts[-1])

# ========================
# Execução
# ========================
rows = []
method_func = get_method_func(method_path)

# Caso 1: função que lista objetos (ex.: all_estimators, all_displays)
if callable(method_func):
    try:
        entries = method_func()
    except TypeError:
        # algumas funções não aceitam chamada direta sem args
        entries = []

    if entries and isinstance(entries[0], tuple):
        # estimators → tuplas (name, class)
        for name, cls in entries:
            full_name = f"{cls.__module__}.{cls.__name__}"
            doc = inspect.getdoc(cls) or ""
            parsed = parse_doc(doc)

            rows.append({
                "name": name,
                "type": "class",
                "full_name": full_name,
                "signature": str(inspect.signature(cls)) if callable(cls) else "",
                "_estimator_type": getattr(cls, "_estimator_type", ""),
                "module": cls.__module__,
                "is_abstract": inspect.isabstract(cls),
                "summary": parsed["summary"],
                "parameters": parsed["parameters"],
                "attributes": parsed["attributes"],
                "see_also": parsed["see_also"],
                "examples": parsed["examples"],
                "other": parsed["other"]
            })
    else:
        # outra função listadora → lista simples
        for obj in entries:
            name = getattr(obj, "__name__", str(obj))
            full_name = f"{obj.__module__}.{name}" if hasattr(obj, "__module__") else str(obj)
            doc = inspect.getdoc(obj) or ""
            parsed = parse_doc(doc)

            rows.append({
                "name": name,
                "type": type(obj).__name__,
                "full_name": full_name,
                "signature": str(inspect.signature(obj)) if callable(obj) else "",
                "_estimator_type": "",
                "module": getattr(obj, "__module__", ""),
                "is_abstract": inspect.isabstract(obj) if inspect.isclass(obj) else False,
                "summary": parsed["summary"],
                "parameters": parsed["parameters"],
                "attributes": parsed["attributes"],
                "see_also": parsed["see_also"],
                "examples": parsed["examples"],
                "other": parsed["other"]
            })

# Caso 2: módulo (ex.: sklearn.metrics)
elif inspect.ismodule(method_func):
    for method in [m for m in dir(method_func) if not m.startswith("_")]:
        obj = getattr(method_func, method)
        if inspect.isclass(obj):
            obj_type = "class"
        elif callable(obj):
            obj_type = "function"
        else:
            obj_type = type(obj).__name__

        name = getattr(obj, "__name__", method)
        full_name = f"{method_func.__name__}.{method}"
        doc = inspect.getdoc(obj) or ""
        try:
            sig = str(inspect.signature(obj)) if callable(obj) else ""
        except (ValueError, TypeError):
            sig = ""

        parsed = parse_doc(doc)

        rows.append({
            "name": name,
            "type": obj_type,
            "full_name": full_name,
            "signature": sig,
            "_estimator_type": getattr(obj, "_estimator_type", ""),
            "module": getattr(obj, "__module__", ""),
            "is_abstract": inspect.isabstract(obj) if inspect.isclass(obj) else False,
            "summary": parsed["summary"],
            "parameters": parsed["parameters"],
            "attributes": parsed["attributes"],
            "see_also": parsed["see_also"],
            "examples": parsed["examples"],
            "other": parsed["other"]
        })

# ========================
# Salvar em TSV
# ========================
df = pd.DataFrame(rows)
df.to_csv(output_file, sep="\t", index=False, encoding="utf-8")

print(f"Arquivo salvo: {output_file} ({len(df)} registros)")
df.head()


Arquivo salvo: all_estimators.tsv (207 registros)


,name,type,full_name,signature,estimator_type,module,is_abstract,summary,parameters,attributes,see_also,examples,other
0,ARDRegression,class,sklearn.linear_model._bayes.ARDRegression,"(*, max_iter=300, tol=0.001, alpha_1=1e-06, al...",regressor,sklearn.linear_model._bayes,False,Bayesian ARD regression.\n\nFit the weights of...,"max_iter : int, default=300\n Maximum numbe...","coef_ : array-like of shape (n_features,)\n ...",BayesianRidge : Bayesian ridge regression.\n\n...,>>> from sklearn import linear_model\n>>> clf ...,
1,AdaBoostClassifier,class,sklearn.ensemble._weight_boosting.AdaBoostClas...,"(estimator=None, *, n_estimators=50, learning_...",classifier,sklearn.ensemble._weight_boosting,False,An AdaBoost classifier.\n\nAn AdaBoost [1]_ cl...,"estimator : object, default=None\n The base...",estimator_ : estimator\n The base estimator...,AdaBoostRegressor : An AdaBoost regressor that...,>>> from sklearn.ensemble import AdaBoostClass...,
2,AdaBoostRegressor,class,sklearn.ensemble._weight_boosting.AdaBoostRegr...,"(estimator=None, *, n_estimators=50, learning_...",regressor,sklearn.ensemble._weight_boosting,False,An AdaBoost regressor.\n\nAn AdaBoost [1] regr...,"estimator : object, default=None\n The base...",estimator_ : estimator\n The base estimator...,AdaBoostClassifier : An AdaBoost classifier.\n...,>>> from sklearn.ensemble import AdaBoostRegre...,
3,AdditiveChi2Sampler,class,sklearn.kernel_approximation.AdditiveChi2Sampler,"(*, sample_steps=2, sample_interval=None)",,sklearn.kernel_approximation,False,Approximate feature map for additive chi2 kern...,"sample_steps : int, default=2\n Gives the n...",n_features_in_ : int\n Number of features s...,SkewedChi2Sampler : A Fourier-approximation to...,>>> from sklearn.datasets import load_digits\n...,
4,AffinityPropagation,class,sklearn.cluster._affinity_propagation.Affinity...,"(*, damping=0.5, max_iter=200, convergence_ite...",clusterer,sklearn.cluster._affinity_propagation,False,Perform Affinity Propagation Clustering of dat...,"damping : float, default=0.5\n Damping fact...",cluster_centers_indices_ : ndarray of shape (n...,AgglomerativeClustering : Recursively merges t...,>>> from sklearn.cluster import AffinityPropag...,


In [8]:
help(estimator_class)

Help on class VotingRegressor in module sklearn.ensemble._voting:

class VotingRegressor(sklearn.base.RegressorMixin, _BaseVoting)
 |  VotingRegressor(estimators, *, weights=None, n_jobs=None, verbose=False)
 |
 |  Prediction voting regressor for unfitted estimators.
 |
 |  A voting regressor is an ensemble meta-estimator that fits several base
 |  regressors, each on the whole dataset. Then it averages the individual
 |  predictions to form a final prediction.
 |
 |  For a detailed example, refer to
 |  :ref:`sphx_glr_auto_examples_ensemble_plot_voting_regressor.py`.
 |
 |  Read more in the :ref:`User Guide <voting_regressor>`.
 |
 |  .. versionadded:: 0.21
 |
 |  Parameters
 |  ----------
 |  estimators : list of (str, estimator) tuples
 |      Invoking the ``fit`` method on the ``VotingRegressor`` will fit clones
 |      of those original estimators that will be stored in the class attribute
 |      ``self.estimators_``. An estimator can be set to ``'drop'`` using
 |      :meth:`set

In [3]:
with open("all_estimators.tsv", "r", encoding="utf-8") as file:
    for line in file.readlines():
        print(line.strip())

estimator_name	estimator_path	estimator_type
ARDRegression	sklearn.linear_model._bayes.ARDRegression	regressor
AdaBoostClassifier	sklearn.ensemble._weight_boosting.AdaBoostClassifier	classifier
AdaBoostRegressor	sklearn.ensemble._weight_boosting.AdaBoostRegressor	regressor
AdditiveChi2Sampler	sklearn.kernel_approximation.AdditiveChi2Sampler
AffinityPropagation	sklearn.cluster._affinity_propagation.AffinityPropagation	clusterer
AgglomerativeClustering	sklearn.cluster._agglomerative.AgglomerativeClustering	clusterer
BaggingClassifier	sklearn.ensemble._bagging.BaggingClassifier	classifier
BaggingRegressor	sklearn.ensemble._bagging.BaggingRegressor	regressor
BayesianGaussianMixture	sklearn.mixture._bayesian_mixture.BayesianGaussianMixture	DensityEstimator
BayesianRidge	sklearn.linear_model._bayes.BayesianRidge	regressor
BernoulliNB	sklearn.naive_bayes.BernoulliNB	classifier
BernoulliRBM	sklearn.neural_network._rbm.BernoulliRBM
Binarizer	sklearn.preprocessing._data.Binarizer
Birch	sklearn.c